# Baseline 04 — SVTR-CTC

SVTR-inspired visual sequence recognition baseline with spatial/channel mixing blocks and CTC decoding. It calculates the same CER and WER metrics.

In [ ]:
%pip install pandas pillow tqdm matplotlib openpyxl

In [ ]:
import os
import json
import math
import time
import random
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# =========================
# Project and processed data
# =========================
cwd = Path.cwd().resolve()
PROJECT_DIR = cwd
if not (PROJECT_DIR / "data").exists():
    for parent in [cwd] + list(cwd.parents):
        if (parent / "data").exists():
            PROJECT_DIR = parent
            break

DATA_DIR = PROJECT_DIR / "data"
assert DATA_DIR.exists(), f"Could not find data directory. Current path: {cwd}"

# Optional: set this manually if you want a specific split folder.
# Example:
# PREFERRED_SPLIT_DIR = DATA_DIR / "processed_line_full" / "splits"
PREFERRED_SPLIT_DIR = None

def load_jsonl_robust(path):
    path = Path(path)
    rows = []
    decoder = json.JSONDecoder()
    with open(path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    idx = 0
    n = len(content)
    while idx < n:
        while idx < n and content[idx].isspace():
            idx += 1
        if idx >= n:
            break
        obj, end = decoder.raw_decode(content[idx:])
        rows.append(obj)
        idx += end
    return rows

def count_jsonl(path):
    try:
        return len(load_jsonl_robust(path))
    except Exception:
        return 0

def find_split_dir():
    if PREFERRED_SPLIT_DIR is not None:
        p = Path(PREFERRED_SPLIT_DIR)
        assert (p / "line_train.jsonl").exists(), f"Missing train split: {p}"
        return p

    candidates = []
    for train_path in DATA_DIR.rglob("line_train.jsonl"):
        split_dir = train_path.parent
        val_path = split_dir / "line_val.jsonl"
        test_path = split_dir / "line_test.jsonl"
        if not val_path.exists() or not test_path.exists():
            continue

        proc_dir = split_dir.parent if split_dir.name == "splits" else split_dir
        n_train = count_jsonl(train_path)
        n_val = count_jsonl(val_path)
        n_test = count_jsonl(test_path)

        score = n_train + n_val + n_test
        name = str(proc_dir).lower()
        if "processed_line_full" in name:
            score += 1_000_000
        if "processed_psocr" in name:
            score += 900_000
        if "full" in name:
            score += 100_000
        if (proc_dir / "vocab.json").exists():
            score += 10_000

        candidates.append((score, split_dir, proc_dir, n_train, n_val, n_test))

    assert len(candidates) > 0, "No processed line-level split files found under data/."
    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)

    print("Detected split candidates:")
    for score, split_dir, proc_dir, n_train, n_val, n_test in candidates[:10]:
        print(f"score={score} | train/val/test={n_train}/{n_val}/{n_test} | {split_dir}")

    return candidates[0][1]

SPLIT_DIR = find_split_dir()
PROC_DIR = SPLIT_DIR.parent if SPLIT_DIR.name == "splits" else SPLIT_DIR

paths = {
    "train": SPLIT_DIR / "line_train.jsonl",
    "val": SPLIT_DIR / "line_val.jsonl",
    "test": SPLIT_DIR / "line_test.jsonl",
}

print("\nSelected SPLIT_DIR:", SPLIT_DIR)
print("Selected PROC_DIR:", PROC_DIR)
for k, p in paths.items():
    print(k, count_jsonl(p), p)

# =========================
# Vocabulary
# =========================
def normalize_pashto_text(text):
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u200c", "")
    text = text.replace("\u200f", "")
    text = text.replace("\ufeff", "")
    text = " ".join(text.split())
    return text.strip()

vocab_path = PROC_DIR / "vocab.json"
if vocab_path.exists():
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab = json.load(f)
    print("Loaded vocab:", vocab_path)
else:
    print("vocab.json not found. Building vocabulary from train split only.")
    train_records_for_vocab = load_jsonl_robust(paths["train"])
    chars = sorted(set("".join(normalize_pashto_text(r["text"]) for r in train_records_for_vocab)))
    itos_tmp = ["<blank>", "<unk>"] + chars
    stoi_tmp = {ch: i for i, ch in enumerate(itos_tmp)}
    vocab = {"itos": itos_tmp, "stoi": stoi_tmp, "blank_id": 0, "unk_id": 1}
    vocab_path = PROC_DIR / "vocab_train_only_auto.json"
    with open(vocab_path, "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False, indent=2)
    print("Saved auto vocab:", vocab_path)

itos = vocab["itos"]
stoi = vocab["stoi"]
BLANK_ID = vocab["blank_id"]
UNK_ID = vocab["unk_id"]

print("Vocab size:", len(itos))

# =========================
# Image resolver
# =========================
image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}
image_map = {}
for p in DATA_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in image_exts:
        image_map[p.name] = p

def resolve_image_path(path_value):
    p = Path(str(path_value))
    if p.exists():
        return p
    for base in [PROJECT_DIR, DATA_DIR, PROC_DIR]:
        q = base / p
        if q.exists():
            return q
    if p.name in image_map:
        return image_map[p.name]
    raise FileNotFoundError(f"Image not found: {path_value}")

# =========================
# Dataset
# =========================
def encode_text(text):
    return [stoi.get(ch, UNK_ID) for ch in normalize_pashto_text(text)]

def decode_ids(ids):
    chars = []
    for i in ids:
        i = int(i)
        if i == BLANK_ID:
            continue
        if i < 0 or i >= len(itos):
            continue
        ch = itos[i]
        if ch in ["<blank>", "<unk>"]:
            continue
        chars.append(ch)
    return "".join(chars)

def preprocess_image(path, img_height=48, max_width=1024):
    img_path = resolve_image_path(path)
    img = Image.open(img_path).convert("L")
    arr0 = np.array(img)
    if arr0.mean() < 127:
        img = ImageOps.invert(img)

    w, h = img.size
    new_w = max(16, int(round(w * img_height / max(1, h))))
    new_w = min(new_w, max_width)

    resample = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC
    img = img.resize((new_w, img_height), resample)

    arr = np.array(img).astype(np.float32) / 255.0
    arr = (arr - 0.5) / 0.5
    return torch.from_numpy(arr).unsqueeze(0), new_w

class PashtoLineOCRDataset(Dataset):
    def __init__(self, jsonl_path, img_height=48, max_width=1024):
        self.records = load_jsonl_robust(jsonl_path)
        self.records = [r for r in self.records if normalize_pashto_text(r.get("text", ""))]
        self.img_height = img_height
        self.max_width = max_width

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image, width = preprocess_image(rec["image"], self.img_height, self.max_width)
        text = normalize_pashto_text(rec["text"])
        label = torch.tensor(encode_text(text), dtype=torch.long)
        return {"image": image, "width": width, "label": label, "text": text, "path": rec["image"]}

def collate_ocr(batch):
    H = max(b["image"].shape[1] for b in batch)
    W = max(b["image"].shape[2] for b in batch)
    images = torch.full((len(batch), 1, H, W), 1.0, dtype=torch.float32)
    for i, b in enumerate(batch):
        img = b["image"]
        _, h, w = img.shape
        images[i, :, :h, :w] = img

    image_widths = torch.tensor([b["width"] for b in batch], dtype=torch.long)
    labels = [b["label"] for b in batch]
    target_lengths = torch.tensor([len(x) for x in labels], dtype=torch.long)
    targets = torch.cat(labels, dim=0)
    return {
        "images": images,
        "image_widths": image_widths,
        "targets": targets,
        "target_lengths": target_lengths,
        "texts": [b["text"] for b in batch],
        "paths": [b["path"] for b in batch],
    }

# =========================
# Metrics
# =========================
def greedy_decode_ctc(logits, input_lengths):
    pred_ids = logits.argmax(dim=-1).detach().cpu().numpy()
    input_lengths = input_lengths.detach().cpu().numpy()
    texts = []
    for b in range(pred_ids.shape[0]):
        seq = pred_ids[b, :input_lengths[b]]
        collapsed = []
        prev = None
        for idx in seq:
            idx = int(idx)
            if idx != BLANK_ID and idx != prev:
                collapsed.append(idx)
            prev = idx
        texts.append(decode_ids(collapsed))
    return texts

def edit_distance(a, b):
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    prev = list(range(m + 1))
    curr = [0] * (m + 1)
    for i in range(1, n + 1):
        curr[0] = i
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev, curr = curr, prev
    return prev[m]

def cer(pred, gold):
    pred = normalize_pashto_text(pred)
    gold = normalize_pashto_text(gold)
    if len(gold) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return edit_distance(pred, gold) / len(gold)

def wer(pred, gold):
    pred_words = normalize_pashto_text(pred).split()
    gold_words = normalize_pashto_text(gold).split()
    if len(gold_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return edit_distance(pred_words, gold_words) / len(gold_words)

def format_seconds(seconds):
    seconds = float(seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"

print("Common CTC utilities loaded.")

In [ ]:
class SVTRBlock(nn.Module):
    def __init__(self, dim=256, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.local_mix = nn.Conv1d(dim, dim, kernel_size=5, padding=2, groups=dim)
        self.channel_mix = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mlp_ratio, dim),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        # x: [B, T, C]
        y = self.norm1(x)
        y = y.transpose(1, 2)
        y = self.local_mix(y).transpose(1, 2)
        x = x + y
        x = x + self.channel_mix(self.norm2(x))
        return x

class SVTRCTC(nn.Module):
    """
    Lightweight SVTR-inspired baseline:
    CNN patch embedding + stacked spatial/channel mixing blocks + CTC.
    """
    def __init__(self, vocab_size, d_model=256, num_layers=8, rtl=True):
        super().__init__()
        self.rtl = rtl
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.GELU(),
            nn.Conv2d(d_model, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.GELU(),
        )
        self.blocks = nn.ModuleList([SVTRBlock(dim=d_model) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, images, image_widths=None):
        x = self.stem(images)
        x = x.mean(dim=2).permute(0, 2, 1)
        if self.rtl:
            x = torch.flip(x, dims=[1])
        for block in self.blocks:
            x = block(x)
        logits = self.fc(self.norm(x))
        T = logits.size(1)
        if image_widths is None:
            input_lengths = torch.full((images.size(0),), T, dtype=torch.long, device=images.device)
        else:
            input_lengths = torch.div(image_widths.to(images.device), 4, rounding_mode="floor")
            input_lengths = torch.clamp(input_lengths, min=1, max=T)
        return logits, input_lengths

def build_model(vocab_size):
    return SVTRCTC(vocab_size=vocab_size, d_model=256, num_layers=8, rtl=True)

In [ ]:
# =========================
# Training configuration
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

IMG_HEIGHT = 48
MAX_WIDTH = 1024
BATCH_SIZE = 16
EPOCHS = 50
LR = 3e-4
WEIGHT_DECAY = 1e-4
AMP = True
NUM_WORKERS = 0  # change to 2 after verifying everything works

BASELINE_NAME = "svtr_ctc"
OUTPUT_DIR = PROJECT_DIR / "outputs" / f"baseline_{BASELINE_NAME}"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FIGURE_DIR = OUTPUT_DIR / "figures"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
for p in [OUTPUT_DIR, CHECKPOINT_DIR, FIGURE_DIR, PREDICTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

train_ds = PashtoLineOCRDataset(paths["train"], IMG_HEIGHT, MAX_WIDTH)
val_ds = PashtoLineOCRDataset(paths["val"], IMG_HEIGHT, MAX_WIDTH)
test_ds = PashtoLineOCRDataset(paths["test"], IMG_HEIGHT, MAX_WIDTH)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_ocr)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_ocr)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_ocr)

model = build_model(vocab_size=len(itos)).to(DEVICE)
print(model.__class__.__name__)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))
print("Output:", OUTPUT_DIR)

criterion = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))

try:
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP and DEVICE.type == "cuda"))
    autocast_context = lambda: torch.amp.autocast("cuda", enabled=(AMP and DEVICE.type == "cuda"))
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=(AMP and DEVICE.type == "cuda"))
    autocast_context = lambda: torch.cuda.amp.autocast(enabled=(AMP and DEVICE.type == "cuda"))

history = {"train_loss": [], "val_loss": [], "val_cer": [], "val_wer": [], "epoch_time_sec": []}
epoch_rows = []
best_val_cer = float("inf")
best_ckpt_path = CHECKPOINT_DIR / f"best_{BASELINE_NAME}.pt"
last_ckpt_path = CHECKPOINT_DIR / f"last_{BASELINE_NAME}.pt"
excel_path = OUTPUT_DIR / f"{BASELINE_NAME}_training_time.xlsx"

def save_excel(epoch_rows, summary_rows):
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        pd.DataFrame(epoch_rows).to_excel(writer, index=False, sheet_name="Epoch_Timing")
        pd.DataFrame(summary_rows).to_excel(writer, index=False, sheet_name="Summary")

def run_validation(loader):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    all_pred, all_gold = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation", leave=False):
            images = batch["images"].to(DEVICE)
            widths = batch["image_widths"].to(DEVICE)
            targets = batch["targets"].to(DEVICE)
            target_lengths = batch["target_lengths"].to(DEVICE)

            logits, input_lengths = model(images, widths)
            if torch.any(input_lengths < target_lengths):
                continue
            log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
            loss = criterion(log_probs, targets, input_lengths, target_lengths)
            if not torch.isfinite(loss):
                continue
            total_loss += loss.item()
            total_batches += 1
            preds = greedy_decode_ctc(logits, input_lengths)
            all_pred.extend(preds)
            all_gold.extend(batch["texts"])

    mean_loss = total_loss / max(1, total_batches)
    mean_cer = float(np.mean([cer(p, g) for p, g in zip(all_pred, all_gold)])) if all_gold else 1.0
    mean_wer = float(np.mean([wer(p, g) for p, g in zip(all_pred, all_gold)])) if all_gold else 1.0
    return mean_loss, mean_cer, mean_wer

training_start = time.perf_counter()
training_start_clock = time.strftime("%Y-%m-%d %H:%M:%S")
print("Training started:", training_start_clock)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    model.train()
    train_loss = 0.0
    train_batches = 0
    skipped = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        images = batch["images"].to(DEVICE)
        widths = batch["image_widths"].to(DEVICE)
        targets = batch["targets"].to(DEVICE)
        target_lengths = batch["target_lengths"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with autocast_context():
            logits, input_lengths = model(images, widths)
            if torch.any(input_lengths < target_lengths):
                skipped += 1
                continue
            log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
            loss = criterion(log_probs, targets, input_lengths, target_lengths)

        if not torch.isfinite(loss):
            skipped += 1
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_batches += 1

    scheduler.step()

    avg_train_loss = train_loss / max(1, train_batches)
    val_loss, val_cer, val_wer = run_validation(val_loader)

    epoch_time = time.perf_counter() - epoch_start
    elapsed = time.perf_counter() - training_start
    avg_epoch = elapsed / epoch
    remaining = avg_epoch * (EPOCHS - epoch)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_cer"].append(val_cer)
    history["val_wer"].append(val_wer)
    history["epoch_time_sec"].append(epoch_time)

    row = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": val_loss,
        "val_CER": val_cer,
        "val_WER": val_wer,
        "epoch_time_seconds": epoch_time,
        "epoch_time_hms": format_seconds(epoch_time),
        "elapsed_seconds": elapsed,
        "elapsed_hms": format_seconds(elapsed),
        "estimated_remaining_seconds": remaining,
        "estimated_remaining_hms": format_seconds(remaining),
        "skipped_batches": skipped,
        "learning_rate": optimizer.param_groups[0]["lr"],
    }
    epoch_rows.append(row)

    print(
        f"Epoch {epoch}: train_loss={avg_train_loss:.4f}, val_loss={val_loss:.4f}, "
        f"val_CER={val_cer:.4f}, val_WER={val_wer:.4f}, "
        f"epoch_time={format_seconds(epoch_time)}, elapsed={format_seconds(elapsed)}, remaining={format_seconds(remaining)}"
    )

    ckpt = {
        "model": model.state_dict(),
        "vocab": vocab,
        "baseline": BASELINE_NAME,
        "history": history,
        "epoch": epoch,
        "val_cer": val_cer,
        "val_wer": val_wer,
    }
    torch.save(ckpt, last_ckpt_path)
    if np.isfinite(val_cer) and val_cer < best_val_cer:
        best_val_cer = val_cer
        torch.save(ckpt, best_ckpt_path)
        print("Saved best:", best_ckpt_path)

    save_excel(epoch_rows, [{
        "status": "running",
        "baseline": BASELINE_NAME,
        "start_time": training_start_clock,
        "last_epoch_completed": epoch,
        "total_epochs": EPOCHS,
        "elapsed_hms": format_seconds(elapsed),
        "estimated_remaining_hms": format_seconds(remaining),
        "best_val_CER": best_val_cer,
    }])

total_time = time.perf_counter() - training_start
summary = [{
    "status": "finished",
    "baseline": BASELINE_NAME,
    "start_time": training_start_clock,
    "end_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "total_epochs": EPOCHS,
    "total_training_time_seconds": total_time,
    "total_training_time_hms": format_seconds(total_time),
    "average_epoch_time_hms": format_seconds(total_time / max(1, EPOCHS)),
    "best_val_CER": best_val_cer,
    "best_checkpoint": str(best_ckpt_path),
}]
save_excel(epoch_rows, summary)

# Test evaluation
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
test_loss, test_cer, test_wer = run_validation(test_loader)
print("Test Loss:", test_loss)
print("Test CER:", test_cer)
print("Test WER:", test_wer)

# Save predictions
prediction_rows = []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Saving test predictions"):
        images = batch["images"].to(DEVICE)
        widths = batch["image_widths"].to(DEVICE)
        logits, input_lengths = model(images, widths)
        preds = greedy_decode_ctc(logits, input_lengths)
        for path_, gold, pred in zip(batch["paths"], batch["texts"], preds):
            prediction_rows.append({
                "image": path_,
                "gold": gold,
                "prediction": pred,
                "cer": cer(pred, gold),
                "wer": wer(pred, gold),
            })

pred_df = pd.DataFrame(prediction_rows)
pred_csv = PREDICTION_DIR / f"{BASELINE_NAME}_test_predictions.csv"
pred_df.to_csv(pred_csv, index=False, encoding="utf-8-sig")
print("Saved predictions:", pred_csv)
print("Saved training Excel:", excel_path)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train Loss", linewidth=2)
axes[0].plot(history["val_loss"], label="Validation Loss", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title(f"{BASELINE_NAME} Loss")
axes[0].legend()

axes[1].plot(history["val_cer"], label="Validation CER", linewidth=2)
axes[1].plot(history["val_wer"], label="Validation WER", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Error Rate")
axes[1].set_title(f"{BASELINE_NAME} CER/WER")
axes[1].legend()

plt.tight_layout()
fig_path = FIGURE_DIR / f"{BASELINE_NAME}_training_curves_700dpi.png"
plt.savefig(fig_path, dpi=700, bbox_inches="tight")
plt.show()
print("Saved figure:", fig_path)